In [4]:
import requests, os, zipfile, io, json
from datetime import datetime

BASE = "/lakehouse/default/Files/transports"
horodatage = datetime.now().strftime("%Y-%m-%d_%Hh%M")

# ---------- 1. Transports en commun (GTFS) ----------
RESEAUX = {
    "mistral_toulon": "https://s3.eu-west-1.amazonaws.com/files.orchestra.ratpdev.com/networks/rd-toulon/exports/gtfs-complet.zip",
    "rtm_marseille":  "https://app.mecatran.com/utw/ws/gtfsfeed/static/mamp-rtm?apiKey=16421b08630a7d065c6d250051780f484b673659",
    "zou_trains":     "https://www.data.gouv.fr/fr/datasets/r/30782be6-90aa-40bb-940a-cb881a95ad26",
}

for nom, url in RESEAUX.items():
    dossier = f"{BASE}/{nom}"
    os.makedirs(dossier, exist_ok=True)
    reponse = requests.get(url)
    reponse.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(reponse.content)) as z:
        z.extractall(dossier)
        print(f"OK GTFS : {nom} → {len(z.namelist())} fichiers")

# ---------- 2. Vélos et trottinettes partagés (GBFS) ----------
GBFS_PARTAGES = {
    "levelo_marseille": "https://api.omega.fifteen.eu/gbfs/2.2/marseille/en/gbfs.json?&key=MjE0ZDNmMGEtNGFkZS00M2FlLWFmMWItZGNhOTZhMWQyYzM2",
    "voi_marseille":    "https://api.voiapp.io/gbfs/fr/6bb6b5dc-1cda-4da7-9216-d3023a0bc54a/v2/66/gbfs.json",
    "lime_marseille":   "https://data.lime.bike/api/partners/v2/gbfs/marseille/gbfs.json",
    "pony_marseille":   "https://proxy.transport.data.gouv.fr/resource/pony-marseille-gbfs/gbfs.json",
}

for nom, url in GBFS_PARTAGES.items():
    dossier = f"{BASE}/{nom}"
    os.makedirs(dossier, exist_ok=True)
    try:
        gbfs = requests.get(url).json()
        # Le sommaire diffère selon la version GBFS (2.x : par langue / 3.0 : direct)
        if "feeds" in gbfs["data"]:
            feeds = gbfs["data"]["feeds"]
        else:
            feeds = list(gbfs["data"].values())[0]["feeds"]

        for f in feeds:
            data = requests.get(f["url"]).json()
            with open(f"{dossier}/{f['name']}_{horodatage}.json", "w") as fichier:
                json.dump(data, fichier)
        print(f"OK GBFS : {nom} → {[f['name'] for f in feeds]}")
    except Exception as e:
        print(f"ÉCHEC : {nom} → {e}")

OK GTFS : mistral_toulon → 9 fichiers
OK GTFS : rtm_marseille → 9 fichiers
OK GTFS : zou_trains → 8 fichiers
OK GBFS : levelo_marseille → ['gbfs', 'system_information', 'station_status', 'station_information', 'vehicle_types', 'free_bike_status', 'system_pricing_plans', 'geofencing_zones', 'gbfs_versions']
OK GBFS : voi_marseille → ['system_information', 'vehicle_types', 'gbfs', 'gbfs_versions', 'system_alerts', 'system_pricing_plans', 'system_regions', 'free_bike_status', 'geofencing_zones', 'station_information', 'station_status']
OK GBFS : lime_marseille → ['system_information', 'station_information', 'station_status', 'free_bike_status', 'vehicle_types']
OK GBFS : pony_marseille → ['gbfs', 'system_information', 'vehicle_types', 'free_bike_status', 'system_pricing_plans', 'geofencing_zones', 'station_information', 'station_status']
